In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [2]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=5, hidden_size=3, num_layers=1, output_size=1):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

input_size = 5
hidden_size = 3
output_size = 1
seq_len = 8
num_samples = 200
batch_size = 16

X = torch.randn(num_samples, seq_len, input_size)
y = torch.randn(num_samples, output_size)

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [4]:
model = SimpleRNN(input_size, hidden_size, 1, output_size).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

model.train()
for epoch in range(10):
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    sample = torch.randn(1, seq_len, input_size).to(device)
    output = model(sample)
    print(output)

tensor([[0.2090]])
